# Pydantic Complete Reference Guide

This notebook is a clean study version of the Pydantic guide. It combines short explanations with runnable Python examples so you can learn step by step.

## 1. Why Pydantic Exists

Python type hints are useful, but they do not enforce data at runtime. Pydantic solves this by validating input data before your program uses it.

In [1]:
def register_user(name: str, email: str, age: int) -> None:
    birth_year = 2026 - age
    print(f"Registered {name}, born around {birth_year}")

# This looks fine, but Python will not stop bad input.
register_user("Aditi", "aditi@example.com", 28)



Registered Aditi, born around 1998


In [2]:
# This will crash later because age is not really an int.
try:
    register_user("Rohan", "rohan@example.com", "unknown")
except Exception as e:
    print("Error:", e)

Error: unsupported operand type(s) for -: 'int' and 'str'


## 2. Your First Model

A Pydantic model validates data using type hints. It turns messy input into a clean, typed object.

In [3]:
from pydantic import BaseModel, ValidationError

class UserModel(BaseModel):
    name: str
    email: str
    age: int

# Valid input
user = UserModel(name="Alice", email="alice@example.com", age=28)
print(user)

name='Alice' email='alice@example.com' age=28


In [4]:
# Invalid input
try:
    UserModel(name="Bob", email="bob@example.com", age="not a number")
except ValidationError as exc:
    print(exc)

1 validation error for UserModel
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 3. Field Constraints and Custom Validators

You can enforce rules using Field constraints, validators, or special types such as EmailStr and HttpUrl.

In [5]:
from typing import Annotated
from pydantic import BaseModel,Field, EmailStr, HttpUrl, SecretStr, field_validator

class JobApplication(BaseModel):
    full_name: Annotated[str, Field(min_length=2, max_length=100)]
    years_experience: Annotated[int, Field(ge=0, le=50)]
    email: EmailStr
    website: HttpUrl

app = JobApplication(
    full_name="pranay",
    years_experience=5,
    email="rohan@example.com",
    website="https://example.com",
)
print(app)

class UserAccount(BaseModel):
    username: str
    password: SecretStr

account = UserAccount(username="rohan99", password="super-secret-123")
print(account.password)

full_name='pranay' years_experience=5 email='rohan@example.com' website=HttpUrl('https://example.com/')
**********


## 4. Computed Fields and Serialization

Computed fields are values derived from other fields. They make the model more expressive and keep data consistent.

In [6]:
from pydantic import computed_field

class ApplicantProfile(BaseModel):
    full_name: str
    years_experience: int

    @computed_field
    @property
    def experience_tier(self) -> str:
        if self.years_experience < 2:
            return "junior"
        if self.years_experience < 7:
            return "mid"
        return "senior"

profile = ApplicantProfile(full_name="Aditi Sharma", years_experience=6)
print(profile.experience_tier)
print(profile.model_dump())

mid
{'full_name': 'Aditi Sharma', 'years_experience': 6, 'experience_tier': 'mid'}


## 5. Nested Models

Real data is often nested. Pydantic validates nested data automatically when you use one model inside another.

In [7]:
class Address(BaseModel):
    city: str
    state: str
    pin_code: str

class Applicant(BaseModel):
    name: str
    email: str
    address: Address

incoming = {
    "name": "Rohan Mehta",
    "email": "rohan@example.com",
    "address": {
        "city": "Pune",
        "state": "Maharashtra",
        "pin_code": "411001",
    },
}

applicant = Applicant.model_validate(incoming)
print(applicant.address.city)

Pune


## 6. Pydantic Settings

Pydantic Settings lets you load configuration from environment variables and .env files safely.

In [8]:
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")

    api_key: SecretStr | None = None
    max_connections: int = Field(default=100, ge=1, le=1000)
    debug: bool = False

settings = AppSettings()
print(settings.max_connections)

100


## 7. Pydantic with FastAPI

FastAPI uses Pydantic models automatically for request validation and documentation.

In [9]:
try:
    from fastapi import FastAPI
except ImportError:
    FastAPI = None

if FastAPI is not None:
    app = FastAPI()

    class ApplicantRequest(BaseModel):
        full_name: str = Field(min_length=2, max_length=100)
        email: EmailStr
        years_experience: int = Field(ge=0, le=50)

    @app.post("/apply")
    def submit_application(applicant: ApplicantRequest):
        return {"message": f"Received from {applicant.full_name}"}

    print("FastAPI app ready")

## 8. Validating LLM Outputs

One of the best uses of Pydantic is validating responses from language models. This makes AI outputs structured and reliable.

In [10]:
from typing import Literal

class ProductInfo(BaseModel):
    name: str
    price: float
    category: str
    in_stock: bool

raw_response = '{"name": "MacBook Pro", "price": 1999.0, "category": "Electronics", "in_stock": true}'
product = ProductInfo.model_validate_json(raw_response)
print(product)

class TicketClassification(BaseModel):
    category: Literal["billing", "shipping", "technical", "general_question"]
    priority: Literal["low", "medium", "high"]
    sentiment: Literal["positive", "negative", "neutral"]
    summary: str = Field(max_length=200)

name='MacBook Pro' price=1999.0 category='Electronics' in_stock=True


## 9. Final Project Idea: AI Support Ticket Triage

This project combines everything you learned: validation, nested models, computed fields, settings, and AI structured output.

In [11]:
import re

class IncomingTicket(BaseModel):
    message: str = Field(min_length=5, max_length=2000)
    customer_name: str | None = None

    @field_validator("message", mode="before")
    @classmethod
    def redact_emails(cls, value):
        if isinstance(value, str):
            return re.sub(r"[\w.-]+@[\w.-]+\.\w+", "[redacted-email]", value)
        return value

class CustomerInfo(BaseModel):
    name: str | None = None
    order_id: int | None = None

class TriagedTicket(BaseModel):
    category: Literal["billing", "shipping", "technical", "general_question"]
    priority: Literal["low", "medium", "high"]
    sentiment: Literal["positive", "negative", "neutral"]
    summary: str = Field(max_length=200)
    customer: CustomerInfo = Field(default_factory=CustomerInfo)

    @computed_field
    @property
    def sla_hours(self) -> int:
        return {"high": 4, "medium": 24, "low": 72}[self.priority]

ticket = IncomingTicket(message="Hello, my order is delayed. Contact me at a@x.com")
print(ticket.message)

Hello, my order is delayed. Contact me at [redacted-email]
